# Engine benchmark: gravity law performance

Benchmarks the gravity law evaluation time for each available numerical engine
across a range of body counts (2 to 1024).

Uses the `scatter` scenario with a fixed random seed for reproducibility.
Produces a log–log plot suitable for insertion into slide 1.2.2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from teachgrav.benchmark import benchmark_range
from teachgrav.scenarios import ScenarioFactory
from teachgrav.laws.laws import create_law
from teachgrav.engine_support import get_available_engines

In [ ]:
SEED = 42
ENGINES_SKIP = {'python', 'numba'}

# Powers of 2 from 2 to 1024
N_BODIES_RANGE = [int(2**k) for k in np.arange(1, 11)]

# Colours matched to engines
ENGINE_COLORS = {
    'numpy':     '#4878d0',
    'jax-cpu':   '#ee854a',
    'jax-gpu':   '#6acc65',
    'jax-metal': '#d65f5f',
    'mlx-cpu':   '#956cb4',
    'mlx-gpu':   '#8c613c',
    'cupy':      '#dc7ec0',
    'torch-cpu': '#797979',
    'torch-gpu': '#d5bb67',
    'torch-mps': '#82c6e2',
}

available = [e for e in get_available_engines() if e not in ENGINES_SKIP]
print('Engines to benchmark:', available)
print('N bodies range:', N_BODIES_RANGE)

In [ ]:
def benchmark_engine_sizes(engine, n_bodies_range, seed):
    """Return benchmark_range results for *engine* over *n_bodies_range*."""
    def fn_at_size(n_bodies):
        factory = ScenarioFactory(engine, seed=seed)
        system = factory.create_scenario('scatter', n_bodies=n_bodies)
        law = create_law('gravity', factory=factory)
        def run_once():
            return law.law(system)
        return run_once
    return benchmark_range(fn_at_size, n_bodies_range, engine=engine)


data = {}
for engine in available:
    print(f'Benchmarking {engine}...')
    try:
        data[engine] = benchmark_engine_sizes(engine, N_BODIES_RANGE, seed=SEED)
    except Exception as exc:
        print(f'  skipped: {exc}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for engine, results in data.items():
    sizes = [r[0] for r in results]
    times = [r[1] for r in results]
    ax.loglog(sizes, times, 'o-',
              label=engine,
              color=ENGINE_COLORS.get(engine),
              linewidth=2,
              markersize=5)

ax.set_xlabel('Number of bodies', fontsize=13)
ax.set_ylabel('Time per evaluation (s)', fontsize=13)
ax.set_title('Gravity law performance by engine', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(min(N_BODIES_RANGE) * 0.8, max(N_BODIES_RANGE) * 1.3)

fig.tight_layout()
plt.show()

In [ ]:
# Save the figure for use in the slideshow
from pathlib import Path

# Resolve the assets directory relative to this notebook's location.
# __file__ is available when executed as a script; fall back to cwd for
# interactive notebook runs (where __file__ is not defined).
try:
    _here = Path(__file__).resolve().parent
except NameError:
    _here = Path.cwd()

repo_root = _here
for _ in range(10):
    if (repo_root / "slideshow").is_dir():
        break
    repo_root = repo_root.parent

output_path = repo_root / "slideshow" / "assets" / "benchmark_engines.png"
fig.savefig(output_path, dpi=150, bbox_inches='tight')
print(f'Saved to {output_path}')